# Seasonal Naive Baseline Forecasts

This notebook implements and evaluates a seasonally naive baseline model for electricity load forecasting on the UCI dataset. The model predicts future values by using the corresponding value from the previous seasonal period (e.g., same time yesterday for daily seasonality). Evaluation is performed using time series cross-validation across multiple validation folds and client sites.

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from constants import (
    UCI_CLIENT_SITES_TO_VALIDATE,
    UCI_DATA_FREQUENCY_MINUTES,
    UCI_VALIDATION_START,
    UCI_VALIDATION_WINDOW,
)

In [ ]:
# Constants

UCI_DATA_DIR = Path("../../data/uci")
UCI_DATA_FILE_NAME = "preprocessed.pq"
UCI_DATA_PATH = UCI_DATA_DIR / UCI_DATA_FILE_NAME

N_VALIDATION_FOLDS = 10

RESULTS_OUTPUT_DIR = Path("../../results/uci/naive")

In [ ]:
class SeasonalNaiveModel:
    def __init__(self, period: timedelta):
        self.period = period
        self._train_end_time: datetime | None = None
        self._train_data: pl.DataFrame | None = None
        self._target_col: str | None = None

    @property
    def is_fit(self) -> bool:
        return (
            self._train_end_time is not None
            and self._train_data is not None
            and self._target_col is not None
        )
    
    def fit(
        self,
        train_data: pl.DataFrame,
        timestamp_col: str = "timestamp",
        target_col: str = "demand",
    ):
        """
        Fit the model on training data by storing the last period of observations.
        
        :param train_data: Training dataframe with timestamp and target columns.
        :param timestamp_col: Name of the timestamp column (default: "timestamp").
        :param target_col: Name of the target column to forecast (default: "demand").
        """
        assert timestamp_col in train_data.columns
        assert target_col in train_data.columns
        
        self._train_end_time = train_data[timestamp_col].max()
        self._target_col = target_col
        self._train_data = (
            train_data
            .filter(pl.col(timestamp_col) >= self._train_end_time - self.period)
            .sort(by=timestamp_col)
        )


    def predict(self, timestamps: pl.Series) -> pl.DataFrame:
        """
        Get forecasts using values from the previous seasonal period.
        
        :param timestamps: Series of timestamps to forecast for.
        :return: DataFrame with timestamp and forecast columns.
        """
        assert self.is_fit
        
        naive_forecasts = (
            timestamps
            .to_frame(name="timestamp")
            # For each forecast timestamp calculcate full number of periods since train_end_time
            .with_columns(
                n_periods=((pl.col("timestamp") - self._train_end_time).dt.total_seconds() / self.period.total_seconds()).ceil()
            )
            # Calculate last observed seasonal timestamp for each forecast timestamp
            .with_columns(
                last_observed=(pl.col("timestamp") - pl.col("n_periods") * self.period)
            )
            # Join on training data and return demand from last observed timestamp.
            .join(
                other=self._train_data, left_on="last_observed", right_on="timestamp", how="left"
            )
            .select(
                pl.col("timestamp"), pl.col(self._target_col)
            )
        )
        
        return naive_forecasts


In [ ]:
# Load preprocessed data

uci_df = pl.read_parquet(UCI_DATA_PATH)
UCI_CLIENT_DF = uci_df.filter(pl.col("client").is_in(UCI_CLIENT_SITES_TO_VALIDATE))

In [ ]:
for client in UCI_CLIENT_SITES_TO_VALIDATE:
    client_results_dir = RESULTS_OUTPUT_DIR / client
    client_results_dir.mkdir(parents=True, exist_ok=True)
    
    client_df = UCI_CLIENT_DF.filter(pl.col("client") == client)
    
    for k in range(N_VALIDATION_FOLDS):
        val_start = UCI_VALIDATION_START + k * UCI_VALIDATION_WINDOW
        val_end = val_start + UCI_VALIDATION_WINDOW
        train_df = (
            client_df
            .filter(pl.col("timestamp").lt(val_start))
            .select(pl.col("timestamp"), pl.col("demand"))
            .sort(by="timestamp")
        )
        val_df = (
            client_df
            .filter(pl.col("timestamp").is_between(val_start, val_end, closed="left"))
            .select(pl.col("timestamp"), pl.col("demand"))
            .sort(by="timestamp")
        )

        naive_model = SeasonalNaiveModel(period=timedelta(hours=24))
        naive_model.fit(train_data=train_df)
        y_hat = naive_model.predict(timestamps=val_df["timestamp"])
        
        # Save forecasts
        forecast_output_path = client_results_dir / f"forecasts_{client}_fold_{k}.pq"
        forecasts_df = val_df.with_columns(forecast=y_hat["demand"])
        forecasts_df.to_pandas().to_parquet(forecast_output_path)

        # Plot forecasts and save
        fig, ax = plt.subplots()
        
        ax.plot(forecasts_df["timestamp"], forecasts_df["demand"], color="black", lw=2, label="Actual")
        ax.plot(forecasts_df["timestamp"], forecasts_df["forecast"], color="#0072B2", lw=2, label="Forecast")
        
        ax.legend(loc=1)
        ax.grid(True, which="major", c="grey", ls="--", lw=1, alpha=0.2)
        ax.set(ylabel="Load (kWh)", title=f"Electricity Load Forecasts for Site {client}, Fold {k}")
        
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
        ax.tick_params(axis='x', labelrotation=45)
        
        fig.tight_layout()
        plt_save_path = client_results_dir / f"forecasts_{client}_fold_{k}.png"
        plt.savefig(plt_save_path, dpi=300)
        plt.close(fig);